# House Price Prediction — Ames Housing Dataset

Predicting residential home sale prices using the Kaggle
["House Prices - Advanced Regression Techniques"](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques)
competition dataset.

**Workflow:** data cleaning → encoding → EDA → feature scaling → model comparison (Ridge, Lasso, Random Forest, Gradient Boosting) with 5-fold cross-validated `GridSearchCV` → final evaluation → Kaggle submission.


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error

pd.set_option('display.max_columns', 50)


## 2. Load Data

In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
test_ids = test['Id']

train = train.drop('Id', axis=1)
test = test.drop('Id', axis=1)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()


## 3. Combine Train + Test Before Encoding

Train and test are combined **before** any encoding so that one-hot encoding
(`pd.get_dummies` with `drop_first=True`) produces identical columns for both sets.
Encoding them separately can silently zero out a feature in one set if a rare
category only appears in the other (a real bug caught and fixed during this project —
see the note in the final section).

In [ ]:
train['dataset'] = 'train'
test['dataset'] = 'test'
combined = pd.concat([train, test], axis=0, ignore_index=True)
combined.shape


## 4. Missing Value Handling

Missing values fall into three groups:
1. **"No feature" columns** — `NaN` means the house doesn't have that feature (e.g. no pool, no garage) → fill with `"None"` / `0`
2. **Genuinely missing numeric/categorical data** — filled with median / mode
3. **Leftover test-set-only quirks** — a few columns had no missing values in train but did in test; filled the same way, using train's statistics only (never test's own, to avoid leakage)

In [ ]:
# Group 1: "no feature" categorical columns
cols_none = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
             'GarageQual', 'GarageFinish', 'GarageType', 'GarageCond',
             'BsmtFinType2', 'BsmtExposure', 'BsmtCond', 'BsmtQual',
             'BsmtFinType1', 'MasVnrType']
for col in cols_none:
    combined[col] = combined[col].fillna('None')

# Group 2: "no feature" numeric columns
combined['GarageYrBlt'] = combined['GarageYrBlt'].fillna(0)
combined['MasVnrArea'] = combined['MasVnrArea'].fillna(0)

# Group 3: genuinely missing
combined['LotFrontage'] = combined['LotFrontage'].fillna(combined['LotFrontage'].median())
combined['Electrical'] = combined['Electrical'].fillna(combined['Electrical'].mode()[0])

# leftover quirks (numeric -> median, categorical -> mode)
for col in combined.select_dtypes(include='number').columns:
    if combined[col].isnull().sum() > 0:
        combined[col] = combined[col].fillna(combined[col].median())
for col in combined.select_dtypes(include='object').columns:
    if col != 'dataset' and combined[col].isnull().sum() > 0:
        combined[col] = combined[col].fillna(combined[col].mode()[0])

print("Missing values remaining:", combined.drop(columns=['dataset']).isnull().sum().sum())


## 5. Encoding

- **Ordinal columns** (natural rank, e.g. quality scales) → mapped to numbers preserving order
- **Nominal columns** (no natural order, e.g. `Neighborhood`) → one-hot encoded

In [ ]:
# shared Ex/Gd/TA/Fa/Po quality scale
qual_map = {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5, 'None': 0}
qual_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
             'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']
for col in qual_cols:
    combined[col] = combined[col].map(qual_map)

# individually-scaled ordinal columns
combined['LotShape'] = combined['LotShape'].map({'Reg': 4, 'IR1': 3, 'IR2': 2, 'IR3': 1})
combined['Utilities'] = combined['Utilities'].map({'AllPub': 4, 'NoSewr': 3, 'NoSeWa': 2, 'ELO': 1})
combined['LandSlope'] = combined['LandSlope'].map({'Gtl': 3, 'Mod': 2, 'Sev': 1})
combined['BsmtExposure'] = combined['BsmtExposure'].map({'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0})

bsmtfin_map = {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'None': 0}
combined['BsmtFinType1'] = combined['BsmtFinType1'].map(bsmtfin_map)
combined['BsmtFinType2'] = combined['BsmtFinType2'].map(bsmtfin_map)

combined['Functional'] = combined['Functional'].map(
    {'Typ': 8, 'Min1': 7, 'Min2': 6, 'Mod': 5, 'Maj1': 4, 'Maj2': 3, 'Sev': 2, 'Sal': 1})
combined['GarageFinish'] = combined['GarageFinish'].map({'Fin': 3, 'RFn': 2, 'Unf': 1, 'None': 0})
combined['PavedDrive'] = combined['PavedDrive'].map({'Y': 2, 'P': 1, 'N': 0})
combined['Fence'] = combined['Fence'].map({'GdPrv': 4, 'MnPrv': 3, 'GdWo': 2, 'MnWw': 1, 'None': 0})


In [ ]:
nominal_cols = ['MSZoning', 'Street', 'Alley', 'LandContour', 'LotConfig',
                'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
                'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
                'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating',
                'CentralAir', 'Electrical', 'GarageType', 'MiscFeature',
                'SaleType', 'SaleCondition']

combined = pd.get_dummies(combined, columns=nominal_cols, drop_first=True)
combined.shape


## 6. Split Back Into Train / Test

In [ ]:
train_final = combined[combined['dataset'] == 'train'].drop(columns=['dataset'])
test_final = combined[combined['dataset'] == 'test'].drop(columns=['dataset', 'SalePrice'])

train_final.shape, test_final.shape


## 7. Exploratory Data Analysis

A couple of quick checks that shaped the preprocessing decisions above.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(train_final['SalePrice'], bins=50)
axes[0].set_title('SalePrice — Raw (right-skewed)')

axes[1].hist(np.log1p(train_final['SalePrice']), bins=50)
axes[1].set_title('SalePrice — Log-transformed')

plt.tight_layout()
plt.show()


In [ ]:
cols_to_check = ['LotFrontage', 'LotArea', 'GrLivArea', 'SalePrice']

fig, axes = plt.subplots(1, len(cols_to_check), figsize=(16, 5))
for i, col in enumerate(cols_to_check):
    axes[i].boxplot(train_final[col])
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

# Outliers reviewed individually (e.g. LotArea > 150,000) and confirmed to be
# legitimate large properties, not data errors — kept in the dataset.


## 8. Train/Test Split, Log Target, and Scaling

- Target is log-transformed (`log1p`) to correct the right-skew shown above — this was the single
  biggest lever for improving linear model performance.
- `StandardScaler` is fit on the training split only, then applied to both splits (no leakage).

In [ ]:
X = train_final.drop('SalePrice', axis=1)
y = train_final['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train.shape, X_test.shape


## 9. Model Comparison — GridSearchCV (5-Fold Cross-Validation)

Four algorithms are compared fairly on the same CV folds and scoring metric,
each with its own hyperparameter grid. Linear models (Ridge, Lasso) use the
scaled features; tree-based models (Random Forest, Gradient Boosting) use the
raw features, since they don't need scaling.

In [ ]:
ridge_params = {'alpha': [0.1, 1, 5, 10, 20, 50, 100]}
ridge_grid = GridSearchCV(Ridge(), ridge_params, cv=5, scoring='r2', n_jobs=-1)
ridge_grid.fit(X_train_scaled, y_train_log)

print("Ridge best params:", ridge_grid.best_params_)
print("Ridge best CV R2:", ridge_grid.best_score_)


In [ ]:
lasso_params = {'alpha': [0.001, 0.01, 0.1, 1, 10]}
lasso_grid = GridSearchCV(Lasso(max_iter=5000), lasso_params, cv=5, scoring='r2', n_jobs=-1)
lasso_grid.fit(X_train_scaled, y_train_log)

print("Lasso best params:", lasso_grid.best_params_)
print("Lasso best CV R2:", lasso_grid.best_score_)


In [ ]:
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_leaf': [1, 2, 4]
}
rf_grid = GridSearchCV(RandomForestRegressor(random_state=42), rf_params, cv=5, scoring='r2', n_jobs=-1)
rf_grid.fit(X_train, y_train_log)

print("Random Forest best params:", rf_grid.best_params_)
print("Random Forest best CV R2:", rf_grid.best_score_)


In [ ]:
gb_params = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1]
}
gb_grid = GridSearchCV(GradientBoostingRegressor(random_state=42), gb_params, cv=5, scoring='r2', n_jobs=-1)
gb_grid.fit(X_train, y_train_log)

print("Gradient Boosting best params:", gb_grid.best_params_)
print("Gradient Boosting best CV R2:", gb_grid.best_score_)


## 10. Comparison Summary

| Model | Best Params | CV R² |
|---|---|---|
| Ridge | alpha=100 | 0.852 |
| Lasso | alpha=0.01 | 0.853 |
| Random Forest | max_depth=20, min_samples_leaf=2, n_estimators=100 | 0.864 |
| **Gradient Boosting** | learning_rate=0.1, max_depth=3, n_estimators=200 | **0.884** |

Gradient Boosting performs best under fair, cross-validated comparison — it beat both
regularized linear models and Random Forest. Notably, a single 80/20 split (rather than CV)
had earlier suggested Ridge was the strongest model (R²≈0.92) — cross-validation revealed that
was partly a lucky split, not a reliable estimate. This is the main reason CV-based comparison
matters more than trusting one split.

In [ ]:
results = pd.DataFrame({
    'Model': ['Ridge', 'Lasso', 'Random Forest', 'Gradient Boosting'],
    'Best CV R2': [ridge_grid.best_score_, lasso_grid.best_score_,
                    rf_grid.best_score_, gb_grid.best_score_]
}).sort_values('Best CV R2', ascending=False)

plt.figure(figsize=(7, 4))
plt.barh(results['Model'], results['Best CV R2'])
plt.xlabel('5-Fold CV R²')
plt.title('Model Comparison')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

results


## 11. Final Evaluation on Held-Out Test Set

In [ ]:
best_model = gb_grid.best_estimator_

final_pred_log = best_model.predict(X_test)
final_pred_actual = np.expm1(final_pred_log)

print("Final Test R2:", r2_score(y_test, final_pred_actual))
print("Final Test RMSE:", mean_squared_error(y_test, final_pred_actual) ** 0.5)


## 12. Feature Importance

Which features the winning model relies on most — useful both for interpretability
and as a sanity check (well-known strong predictors like `OverallQual` and `GrLivArea`
should rank near the top).

In [ ]:
importances = pd.Series(best_model.feature_importances_, index=X_train.columns)
top_features = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 6))
plt.barh(top_features.index[::-1], top_features.values[::-1])
plt.title('Top 15 Feature Importances — Gradient Boosting')
plt.tight_layout()
plt.show()


## 13. Kaggle Submission

In [ ]:
test_pred_log = best_model.predict(test_final)
test_pred_actual = np.expm1(test_pred_log)

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': test_pred_actual})
submission.to_csv('submission.csv', index=False)

submission['SalePrice'].describe()


## Notes / Lessons Learned

- **Cross-validation vs single split:** a single train/test split overstated Ridge's performance
  (R²≈0.92 vs the honest CV estimate of 0.852). Always prefer CV for model comparison.
- **Log-transforming the target** was the single biggest lever for the linear models.
- **Encoding train and test jointly** (before splitting back apart) avoided a real bug where a
  rare category (`RoofMatl_CompShg`) was silently dropped from the test set's one-hot columns
  because it wasn't the same "first" category alphabetically in test as in train.
- **Multicollinearity removal and Lasso-based feature selection** made negligible difference to
  Ridge's accuracy — regularization was already handling redundant features.
